# ဒြပ်ဆင်း ၁၈: ကရစ်ပိုဂရဖစ် စာရင်းများဖြင့် AI ကိုယ်စားလှယ်များကို လုံခြုံစေရန်

## လက်တွေ့အလုပ်အပ်စာအုပ်

ဒီစာအုပ်မှာ လေ့ကျင့်ခန်းလေးချက် ပတ်သတ်ထားတယ် -

1. ကိုယ်စားလှယ်ကိရိယာခေါ်ဆိုမှုအတွက် **ပထမဆုံး စာရင်းကို လက်မှတ်ထိုးပါ**၊ ပြီးတော့ အတည်ပြုပါ။
2. **စာရင်းကို ခိုးစားတည်းဖြတ်ကြည့်ပါ**၊ အတည်ပြုမှု မအောင်မြင်ကြောင်း ကြည့်ပါ။
3. **စာရင်းသုံးခု ချိတ်ဆက်ပြီး စွဲချက်တည်ကြောင်း အတည်ပြုပါ**။
4. **Microsoft Agent Framework ကိရိယာခေါ်ဆိုမှုကို ထုပ်ပိုးပါ**၊ လုပ်ဆောင်မှုတိုင်းမှာ စာရင်း ထုတ်ပေးပါ။

ကုဒ်အခြေခံသော cryptographic primitives များကို ကောင်းစွာထိန်းသိမ်းထားသော စာကြည့်တိုက်များမှ သွင်းယူထားသည် (`pynacl` သည် Ed25519 အတွက်၊ `jcs` သည် RFC 8785 canonical JSON အတွက်၊ `hashlib` သည် Python standard library မှ SHA-256 အတွက်)။ သို့ရာတွင် receipt logic ကိုယ်တိုင်သည် သင်ဖတ်ရန်နှင့် ပြင်ဆင်ရန်ရိုးရှင်းသော Python ဖြစ်သည်။

စာရွက်ချုပ်များကို အစဉ်လိုက် အလုပ်အတည်းလုပ်ပါ။ အပိုင်းတိုတိုတိုင်းသည် ကိုယ်ပိုင်ထားပြီး လွယ်ကူစွာနားလည်နိုင်သည်။


## အတည်ပြုခြင်း

ဥပဒေခွင့်ပြုသည့် လိုင်စင်ဖြင့် (Apache-2.0 / MIT) နှစ်ခုစလုံးပါ သက်ဆိုင်သော Package နှစ်ခုကို တပ်ဆင်ပါ။


In [1]:
%pip install -q pynacl jcs

Note: you may need to restart the kernel to use updated packages.


In [2]:
import json
import hashlib
import base64
from datetime import datetime, timezone

from nacl import signing
from nacl.exceptions import CryptoError
from jcs import canonicalize

## အကူအညီ ကိရိယာများ

ဒီအကူအညီနှစ်ခုက base64url ကုဒ်ပြုလုပ်ခြင်း (padding မပါဘဲ) နဲ့ SHA-256 ဟက်ရှ် ချိတ်ခြင်းကို ကိုင်တွယ်ပေးပါတယ်။ ဒါကြောင့် မှတ်စုစာတမ်းနောက်ကျောမှာ ရှိတာတွေက ရက်အစီအစဉ်နည်းလမ်းမှာသာ ဦးတည်ထားနိုင်ပါတယ်။


In [3]:
def b64url_nopad(data: bytes) -> str:
    """Base64url-encode bytes without padding (RFC 4648 Section 5)."""
    return base64.urlsafe_b64encode(data).decode("ascii").rstrip("=")

def b64url_decode(s: str) -> bytes:
    """
    Decode unpadded base64url strictly, raising ValueError for anything else.

    Python's decoder skips characters outside the alphabet and also accepts
    padding, the standard +/ alphabet and non-zero spare bits, so the same key
    or signature could be written several ways. Re-encoding and comparing
    accepts only the one form that b64url_nopad produces.
    """
    padding = "=" * ((4 - len(s) % 4) % 4)
    data = base64.urlsafe_b64decode(s + padding)
    if b64url_nopad(data) != s:
        raise ValueError("not canonical unpadded base64url")
    return data

def sha256_canonical(obj) -> str:
    """
    SHA-256 hash of a Python object, computed over its JCS-canonical JSON form.
    Returns a 'sha256:' prefixed hex digest so callers can identify the algorithm.
    """
    canonical = canonicalize(obj)
    digest = hashlib.sha256(canonical).hexdigest()
    return f"sha256:{digest}"

## အပိုင်း ၁: သင်၏ ပထမဆုံး လက်မှတ်ကို လက်မှတ်ရေးထိုးပါ

**Contoso Travel** ၏ ကိုယ်စားလှယ်တစ်ဦးသည် စစ်တမ်းပြုလုပ်သူအတွက် Sydney ကနေ Los Angeles သို့ အပန်းဖြေ ခရီးစဉ်ကို လေ့လာပြီးဖြစ်သည် ဆိုင်းငံ့ပါစေ။ ဒီကိရိယာခေါ်ဆိုမှုအပေါ် gateway ၏ ကြေညာချက်ကို မှတ်တမ်းတင်ချင်တာဖြစ်ပြီး အနာဂတ် auditor တစ်ဦးက ဘေးကမ်း issuer key တစ်ခုအောက်မှာ အတည်ပြုနိုင်စေရန်ဖြစ်သည်။

### အဆင့် ၁.၁: လက်မှတ်ရေးထိုးရန် key များဖန်တီးပါ

ပစ္စည်းထုတ်လုပ်မှုတွင် ကိုယ်စားလှယ်၏ လက်မှတ်ရေးထိုး key ကို hardware security module (HSM), Azure Key Vault သို့မဟုတ် ဆင်တူကာကွယ်ထားသော အစုအဝေးတစ်ခုတွင် သိမ်းဆည်းထားပါသည်။ ဒီသင်ခန်းစာအတွက် memory တွင်အသစ်သော key ကို ဖန်တီးပါသည်။


In [4]:
signing_key = signing.SigningKey.generate()
verify_key = signing_key.verify_key

public_key_b64 = b64url_nopad(bytes(verify_key))
# Trust configuration is out of band. Never trust a receipt only because it embeds a key.
TRUSTED_PUBLIC_KEYS = {public_key_b64}
print(f"Public key (Ed25519, 32 bytes): {public_key_b64}")

Public key (Ed25519, 32 bytes): NKiSqaTHCofGAokp-IyN_y6xJH14J5aUTkL64oMK5uM


### အဆင့် ၁.၂: လက်မှတ် payload ကို တည်ဆောက်ပါ

Payload က ငါတို့ လက်မှတ်က ပိုင်ခွင့်ပြုဖို့လိုအပ်တဲ့ အရာအားလုံး ပါဝင်ပါတယ်။ ဘယ်သူ လုပ်ခဲ့သလဲ၊ ဘယ်စက်ကိရိယာ၊ ဘယ်အချက်အလက်တွေနဲ့၊ ဘာတွေပြန်လာခဲ့သလဲ၊ ဘယ်မူဝါဒအောက်မှာ၊ ဘယ်အချိန်လဲဆိုတာ ဖြစ်ပါတယ်။ Argument နဲ့ အကျိုးသက်ရောက်မှုကို တိုက်ရိုက် ထည့်သွင်းရန်မဟုတ်ဘဲ hash လုပ်ခြင်းအားဖြင့် လက်မှတ်မှာ အချက်အလက် sensibil သာမှန်ချက်များ ပေါက်ဖော်ထွက်ပေါက်နေခြင်းကို တားဆီးသည်။


In [5]:
tool_args = {
    "origin": "SYD",
    "destination": "LAX",
    "departure_date": "2026-06-15",
    "passengers": 2,
}

tool_result = [
    {"flight": "QF11", "price": 1850, "stops": 0},
    {"flight": "UA864", "price": 1620, "stops": 1},
    {"flight": "DL11", "price": 1740, "stops": 0},
]

payload = {
    "type": "agent.tool_call.v1",
    "agent_id": "contoso-travel-bot",
    "tool_name": "lookup_flights",
    "tool_args_hash": sha256_canonical(tool_args),
    "result_hash": sha256_canonical(tool_result),
    "policy_id": "contoso-travel-policy-v3",
    "timestamp": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
    "sequence": 0,
    "previous_receipt_hash": None,
}

print(json.dumps(payload, indent=2))

{
  "type": "agent.tool_call.v1",
  "agent_id": "contoso-travel-bot",
  "tool_name": "lookup_flights",
  "tool_args_hash": "sha256:47578acca4df262c8f172b91493b818a26d042e7beb8e7b121e2bc3776152746",
  "result_hash": "sha256:556447bf01c3c33285086d224b3d6fb4cd7b620b5151fbbebe67e7ff81cdde7a",
  "policy_id": "contoso-travel-policy-v3",
  "timestamp": "2026-10-16T04:44:07Z",
  "sequence": 0,
  "previous_receipt_hash": null
}


### အဆင့် ၁.၃: လက်မှတ်ထိုးပြီး လက်မှတ်ထွက်စာကို စုပုံခြင်း

တစ်ချက်ချင်းသုံးဆင့်:

၁။ JCS ကို အသုံးပြုပြီး payload ကို canonicalize ပြုလုပ်ပါ၊ ဒါကတူညီသော လိုဂျစ်ကယ် လက်မှတ်ထွက်စာများကို byte အတူတူဖြစ်အောင် ပြုလုပ်မည်။
၂။ Ed25519 ကိုယ်ပိုင် key ဖြင့် canonical bytes ကိုတိုတိုက်က် အလျော့လိုက် လက်မှတ်ထိုးပါ။ PureEdDSA သည် မက်ဆေ့ခ််ကို အတွင်းပိုင်းတွင် hash လုပ်သည်၊ ထို့ကြောင့် အပို pre-hash လုပ်ခြင်းသည် protocol ကိုပြောင်းလဲစေမည်။

ထို့နောက် အဆိုပါလက်မှတ်ကို မူရင်း payload နှင့် တဆက်တည်း ပို့ဆောင်၍ နောက်ဆုံး လက်မှတ်ထွက်စာကို ထုတ်လုပ်သည်။


In [6]:
def sign_receipt(payload: dict, signing_key: signing.SigningKey, verify_key) -> dict:
    """
    Sign a receipt payload. Returns the receipt with attached signature and public key.
    The 'signature' and 'public_key' fields are NOT part of the canonical signed bytes.
    """
    canonical = canonicalize(payload)
    signature_bytes = signing_key.sign(canonical).signature
    return {
        **payload,
        "signature": {
            "alg": "EdDSA",
            "sig": b64url_nopad(signature_bytes),
            "public_key": b64url_nopad(bytes(verify_key)),
        },
    }

receipt = sign_receipt(payload, signing_key, verify_key)
print(json.dumps(receipt, indent=2))

{
  "type": "agent.tool_call.v1",
  "agent_id": "contoso-travel-bot",
  "tool_name": "lookup_flights",
  "tool_args_hash": "sha256:47578acca4df262c8f172b91493b818a26d042e7beb8e7b121e2bc3776152746",
  "result_hash": "sha256:556447bf01c3c33285086d224b3d6fb4cd7b620b5151fbbebe67e7ff81cdde7a",
  "policy_id": "contoso-travel-policy-v3",
  "timestamp": "2026-10-16T04:44:07Z",
  "sequence": 0,
  "previous_receipt_hash": null,
  "signature": {
    "alg": "EdDSA",
    "sig": "NnH0T6lpjVThg1RS29rEu3OG1FptWbQgX18Yt1JAPjNnTyje2Luozt5_0gcdxddQXoS6eH6Vlg5MTcKLrvWnBQ",
    "public_key": "NKiSqaTHCofGAokp-IyN_y6xJH14J5aUTkL64oMK5uM"
  }
}


### အဆင့် ၁.၄: လက်ခံမှုကို စစ်ဆေးပါ

စစ်ဆေးခြင်းသည် လုပ်ထုံးလုပ်နည်းကို မောင်းလှည့်သည်။ ကျွန်ုပ်တို့သည် လက်မှတ်ကို ဖယ်ရှားပြီး ကန့်သတ်ထားသော bytes များကို ပြန်တွက်ချက်ကာ လက်မှတ်အား ထုတ်သူ၏ အများပြည်သူသော key နှင့် စစ်ဆေးသည်။

စစ်ဆေးခြင်းသည် အော့ဖ်လိုင်းဖြစ်သည်- ဝန်ဆောင်မှုကို ခေါ်ရန်မလိုနှင့် ကွန်ယက်အသုံးပြုခွင့်မရှိပါ။ ထိုသို့ဆိုလျှင်လက်ခံမှုအပြင် တခြားတစ်ခုလိုအပ်သည်။ ထိုသို့သတ်မှတ်ထားသော စနစ်ဖြင့် စစ်ဆေးသူ ယုံကြည်နိုင်သော ချန်နယ်မှ ရရှိသော ထုတ်သူ၏ အများပြည်သူသော key ဖြစ်သည်။ ဥပမာ- ထုတ်သူမှ ထုတ်ဝေသော JWK စုစည်းမှု သို့မဟုတ် ကြိုတင်မှတ်ထားထားသောတန်ဖိုးဖြစ်နိုင်သည်။ လက်ခံမှုတွင် ပါဝင်သော `public_key` ကွက်လပ်သည် ထိုအခန်းကဏ္ဍ မမြဲနိုင်ပါ၊ အဲနေ့ဘယ်သူရိုက်ထုတ်လဲဆိုတာမှ လက်ခံမှုရေးသားသူနှင့် အတူတူဖြစ်သဖြင့် ဖြစ်သည်။ အပိုင်း ၂ တွင် ဤကာကွယ်မှု၏ လုပ်ဆောင်ပုံကို ပြသထားသည်။


In [7]:
def verify_receipt(receipt: dict, trusted_public_key: str) -> bool:
    """
    Verify a receipt's Ed25519 signature against a key you already trust.

    trusted_public_key is the issuer's key, obtained from somewhere other than
    this receipt: a JWK Set, a pinned config value, a key registry. Reading the
    key out of receipt["signature"]["public_key"] instead would only prove the
    receipt is consistent with itself. Section 2 shows exactly how that fails.

    It fails closed: anything malformed returns False instead of raising, so
    one bad receipt cannot stop a batch of them from being checked.
    """
    if not isinstance(receipt, dict) or not isinstance(trusted_public_key, str):
        return False
    sig_obj = receipt.get("signature")
    if not isinstance(sig_obj, dict) or sig_obj.get("alg") != "EdDSA":
        return False
    sig_b64 = sig_obj.get("sig")
    if not isinstance(sig_b64, str):
        return False

    try:
        # Reconstruct the payload that was actually signed (everything except signature).
        payload = {k: v for k, v in receipt.items() if k != "signature"}
        canonical = canonicalize(payload)
        verify_key = signing.VerifyKey(b64url_decode(trusted_public_key))
        verify_key.verify(canonical, b64url_decode(sig_b64))
        return True
    except CryptoError:
        return False
    except Exception:
        # An undecodable key or signature, or a payload that cannot be
        # canonicalized: fail closed rather than raise.
        return False

# public_key_b64 came from Section 1, alongside the signing key. In a real
# deployment it would come from the issuer's JWK Set, not from this notebook.
is_valid = verify_receipt(receipt, public_key_b64)
print(f"Receipt is valid: {is_valid}")

# Regression control for the signature scope in draft revision 02. A receipt
# signed over SHA-256(JCS(payload)) is a signature over different bytes and
# must not verify as a direct-JCS Ed25519 receipt.
prehashed_signature = signing_key.sign(hashlib.sha256(canonicalize(payload)).digest()).signature
prehashed_receipt = {
    **receipt,
    "signature": {**receipt["signature"], "sig": b64url_nopad(prehashed_signature)},
}
print(f"Pre-hashed receipt valid: {verify_receipt(prehashed_receipt, public_key_b64)}")

# Malformed input is rejected, not raised.
for malformed in ({"signature": "x"}, {"signature": {"alg": "EdDSA"}}, "not a receipt"):
    assert verify_receipt(malformed, public_key_b64) is False
print("Malformed receipts rejected: True")

Receipt is valid: True
Pre-hashed receipt valid: False
Malformed receipts rejected: True


သင်သည် `Receipt is valid: True`၊ `Pre-hashed receipt valid: False` နှင့် `Malformed receipts rejected: True` ကို မြင်ရမည်။ အကြောင်းပြချက်ကောင်းသည် တိုက်ရိုက်-JCS လက်မှတ်ထိုးလမ်းကြောင်းသည် အလုပ်လုပ်ကြောင်း သက်သေပြသည်။ မကောင်းသော ထိန်းချုပ်မှုက လက်မှတ်-အကွက်စည်းမျဉ်းကို ပရိုဇ်အနေဖြင့်မထားဘဲ တည်ဆောင်နိုင်စေသည်။ နောက်ဆုံးကြောင်းသည် မမှန်ကန်သော အချက်အလက်ဖြင့် `False` ကို ပြန်လည်ပေးပို့ခြင်းဖြစ်ပြီး အမှားထုတ်ပြန့်မှုမဖြစ်စေပါ။


## အပိုင်း ၂ - လက်ခံရရှိချက်ကို ဖျက်ပြင်ခြင်း

လက်ခံရရှိချက်ရဲ့ အဓိကအချက်ကတော့ ဖျက်ပြင်မှုတွေ ထင်ရှားပေါ်နေဖို့ ဖြစ်ပါတယ်။ ပိုင်းအစမ်းပြကြမယ်။

လက်ခံရရှိချက်အတွင်းမှာ စာလုံး တစ်လုံးတည်းကို တိတိကျကျ ပြင်ဆင်ပြီး စစ်ဆေးမှု မအောင်မြင်တာကို ကြည့်မယ်။


In [8]:
import copy

tampered = copy.deepcopy(receipt)

# Modify the policy_id field (this is what an attacker might do to claim
# the action was governed by a more permissive policy than was actually used).
original_policy = tampered["policy_id"]
tampered["policy_id"] = "contoso-travel-policy-PERMISSIVE"

print(f"Original policy_id:  {original_policy}")
print(f"Tampered policy_id:  {tampered['policy_id']}")
print()
print(f"Tampered receipt valid? {verify_receipt(tampered, public_key_b64)}")

Original policy_id:  contoso-travel-policy-v3
Tampered policy_id:  contoso-travel-policy-PERMISSIVE

Tampered receipt valid? False


### ဘာဖြစ်သွားတာလဲ?

ကျွန်ုပ်တို့ `policy_id` ကိုပြောင်းလဲသောအခါ၊ canonical bytes က ပြောင်းလဲသွားသည်။ အစီရင်ခံချက် (မူလ canonical bytes များအပေါ်တွင်ရှိသော) သုံး၍လက်မှတ်သည် သက်ဆိုင်မှုမရှိတော့ပါ။ အတည်ပြုမှုမှန်ကန်စွာ `False` ကိုပြန်လည်ထုတ်ပေးသည်။

လက်မှတ်ရေးထားသော filed ကိုပြောင်းလဲရန် မရှိပါ၊ အသက်တာသည် `signature` object ပြင်ပတွင်ရှိသည်ကိုဆိုလိုသည်၊ issuer ၏ လူမှုသော key နှင့် receipt အတည်ပြုမှု ပြုလုပ်နိုင်ခြင်းမရှိပါ၊ attacker မှ issuer ၏ ကိုယ်ပိုင် key မရှိပါကသာ ဖြစ်နိုင်သည်။ `signature` object ကိုယ်တိုင်သည် လက်မှတ်ရေးထားခြင်းမရှိဘဲဖြစ်သည်။ `sig` သည် လက်မှတ်ဖြစ်ပြီး `verify_receipt` သည် `public_key` ကို မမျှော်လင့်ပါ၊ ထို filed ကိုတစ်ခုတည်း ပြင်ဆင်ခြင်းက မပြောင်းလဲစေပါ။ ကိုယ်ပိုင် key ကို key vault တွင်သေချာစွာထားရှိပြီး verifier များသည် issuer ၏ သက်သေပြုထားသော အများပြည်သူ key ကို အသုံးပြုပါက လက်မှတ်ရေးထားသော filed ပြောင်းလဲကာ လျှို့ဝှက်ထားရန် မဖြစ်နိုင်ပါ။ ထိုအခြေအနေမှာ အရေးကြီးသည်။ နောက်ထပ် cell တစ်ခုတွင် မူလ key မတူညီသော receipt အတည်ပြုမှုကို ပြသသည်။

ကိုယ်တိုင် စမ်းသပ်ကြည့်ပါ: အပေါ်က cell တွင် `tool_name` သို့မဟုတ် `agent_id` သို့မဟုတ် `timestamp` ကို ပြင်ဆင်ပြီး ထပ်မံ run မည်။ လက်မှတ်ရေးထားသော filed တစ်ခုလုံးကိုပြောင်းလဲတတ်ခြင်းအားလုံးသည် အတည်ပြုမှု မရှိသော receipt ဖြစ်စေပါသည်။


### လက်မှတ်က ဖမ်းဆီးမရတဲ့ မတရားပြင်ဆင်မှု

အထက်ဖော်ပြပါတိုက်ခိုက်သူသည် ပြန်လည်စာချုပ်မထိုးခဲ့ပါ။ သေချာစွာလုပ်တတ်သူတစ်ဦးမှာတော့ - ငွေလက်ခံရေးလက်မှတ်ကိုပြင်ဆင်မည်လိမ့်မည်၊
သူတို့ကိုယ်ပိုင်ဖန်တီးထားသော key ဖြင့် အမှတ်ဆိပ်ထိုးပြီး ထို key ကို
`public_key` ကွက်ထဲတွင်ထည့်ပါ။ ရလဒ်မှာ လက်မှတ်သည် အစုံစုံမှန်မှန်ဖြစ်သော လက်ခံရရှိမှုဖြစ်သည်
၎င်း၏အကြောင်းအရာများအတွက်။

ဒါကို chạy ကြည့်ပါ၊ ဘယ်တိုင်းတာနှစ်ခုထဲမှ ဘယ်ဟာက ဒါကို ဖမ်းဆီးနိုင်တယ်ဆိုတာ ကြည့်ပါ။


In [9]:
# The attacker generates their own keypair. Nothing stops them.
attacker_key = signing.SigningKey.generate()

forged_payload = {k: v for k, v in receipt.items() if k != "signature"}
forged_payload["policy_id"] = "contoso-travel-policy-PERMISSIVE"

# Re-signed, with the attacker's key embedded where the issuer's used to be.
forged = sign_receipt(forged_payload, attacker_key, attacker_key.verify_key)

print(f"Issuer's public key : {public_key_b64}")
print(f"Key inside the forgery: {forged['signature']['public_key']}")
print()
print(f"Checked against the key inside it: "
      f"{verify_receipt(forged, forged['signature']['public_key'])}")
print(f"Checked against the issuer's key : "
      f"{verify_receipt(forged, public_key_b64)}")

# Assert the results, so the cell fails if the forgery ever passes as the issuer's.
assert verify_receipt(receipt, public_key_b64) is True                    # genuine, issuer's key
assert verify_receipt(forged, forged["signature"]["public_key"]) is True  # forgery, its own key
assert verify_receipt(forged, public_key_b64) is False                    # forgery, issuer's key

Issuer's public key : NKiSqaTHCofGAokp-IyN_y6xJH14J5aUTkL64oMK5uM
Key inside the forgery: nIXlWGm0syJQysGkn-nRlzFS59HlPCTqeQr32TJoHM4

Checked against the key inside it: True
Checked against the issuer's key : False


သင်သည် `True` ထပ်မံပြီးနောက် `False` ကို တွေ့ရမည်။ သုံးကြောင်းသော `assert` ပုဒ်မများသည်
စစ်ဆေးမှုတစ်ခု ဖြစ်စေသည်။ စွန့်စားမှုသည် issuer ၏ key အပေါ်တွင် အတည်ပြုခဲ့လျှင် cell သည် error တက်စေပါသည်။


စွန့်စားမှုသည် အတွင်းပိုင်းထဲတွင် သင့်တော်မှုရှိသည်။ ၎င်း၏ အမှတ်တံဆိပ်သည် ၎င်း၏
အကြောင်းအရာများနှင့် ၎င်းပါသော key နှင့် လုံးဝကိုက်ညီသည်၊ ထို့ကြောင့် key ကို
receipt ထဲမှ ဖတ်ရှုသော verifier သည် ၎င်းကို တရားဝင်ဟု အသိပေးသည်။ receipt အကြောင်း
လုံးဝ မှားယွင်းခြင်း မရှိပါ၊ အကြောင်းအရင်းမှာ ၎င်းသည် မညှိမတည့်မှု များ မရှိပါ။

တစ်ခုတည်းပဲ ယင်းကို ဖမ်းဆီးမည့် နည်းလမ်းမှာ attacker အနေနဲ့ ထိခိုက်နိုင်ခြင်း မရှိသော key၊
နှင့် နှိုင်းယှဉ်ခြင်း ဖြစ်သည်။ ထို့ကြောင့် `verify_receipt` သည် key ကိုအကြောင်းအရာအဖြစ် လက်ခံသည်။ 
`verify_receipt(receipt)` အမှတ်တံဆိပ်ရှိသော function ကို လုံခြုံစွာ ရေးသား၍ မရပါ၊
အကြောင်းက သာတိုက်ရိုက်ရောက်နိုင်သော key သည် attacker ပေးပို့လိုက်သော key တစ်ခုသာဖြစ်သောကြောင့်ဖြစ်သည်။

အဲဒီ receipt ကို `sample_receipts/04_resigned_receipt.json` အဖြစ် မှတ်တမ်းတင်ထားသည်၊
သင်အွန်ဘုတ်ပြား အပြင်မှာ စစ်ဆေးလိုပါက။


## အပိုင်း ၃: လက်မှတ်များကို ဆက်သွယ်ပါ

တစ်ချက်လက်မှတ်သည် တစ်ခုတည်းသော လုပ်ဆောင်ချက်တစ်ခုကိုကာကွယ်သည်။ အများစုအေးဂျင့်များသည် လုပ်ဆောင်ချက်များစွာ ကိုင်တွယ်သည်။ စုစည်းမှုလုံးဝကို အတုခြစ်မှု အထောက်အထားရှိအောင် ပြုလုပ်ရန်၊ လက်မှတ်အသစ်၏ ပေးလုပ်ခွင့်တွင် ယခင်လက်မှတ်၏ ဟက်ရှ်အား ထည့်သွင်းခြင်းဖြင့် လက်မှတ်တိုင်းကို ယခင်လက်မှတ်နှင့် ဆက်သွယ်ပါသည်။

```text
Receipt 0  -->  Receipt 1  -->  Receipt 2
                  |                 |
                  +-- previous_receipt_hash field --+
```

တင်ပြထားသည့် စုပြုံမှုအတွင်းမှ မည်သူမဆို လက်မှတ်တစ်ခုကို ဖယ်ရှားခြင်း သို့မဟုတ် နေရာပြောင်းခြင်း ပြုလုပ်ပါက နောက်လိပ်ချိတ် မှတ်တမ်းပျက်ကွက်သွားပါမည်။ စာရင်းဖြတ်လမ်း ခွဲခြားခြင်း သို့မဟုတ် လက်မှတ်ရေးရန်ထိန်းချုပ်သူနောက်ဆက်တွဲပြင်ဆင်မှုများကို သိရှိရန်လည်း စည်းစိမ်ထားသော လက်ရှိခေါင်းစဉ်၊ အပြင်ပြိုင်စစ်ဆေးမှု သို့မဟုတ် အတည်ပြုထားသော လူဦးရေကတိ တစ်ခုရှိရန် လိုအပ်ပါသည်။


In [10]:
def receipt_hash(receipt: dict) -> str:
    """
    Compute the chain hash of a complete receipt (including signature).
    This becomes the previous_receipt_hash of the next receipt in the chain.
    """
    canonical = canonicalize(receipt)
    digest = hashlib.sha256(canonical).hexdigest()
    return f"sha256:{digest}"

def make_receipt(
    tool_name: str,
    tool_args: dict,
    tool_result,
    sequence: int,
    previous_receipt_hash,
    signing_key,
    verify_key,
    agent_id: str = "contoso-travel-bot",
    policy_id: str = "contoso-travel-policy-v3",
) -> dict:
    """Convenience: build, sign, and return a receipt for one tool call."""
    payload = {
        "type": "agent.tool_call.v1",
        "agent_id": agent_id,
        "tool_name": tool_name,
        "tool_args_hash": sha256_canonical(tool_args),
        "result_hash": sha256_canonical(tool_result),
        "policy_id": policy_id,
        "timestamp": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
        "sequence": sequence,
        "previous_receipt_hash": previous_receipt_hash,
    }
    return sign_receipt(payload, signing_key, verify_key)

In [11]:
# Build a chain of three receipts: search, hold, book.
r0 = make_receipt(
    tool_name="lookup_flights",
    tool_args={"origin": "SYD", "destination": "LAX", "date": "2026-06-15"},
    tool_result=[{"flight": "QF11", "price": 1850}],
    sequence=0,
    previous_receipt_hash=None,
    signing_key=signing_key,
    verify_key=verify_key,
)

r1 = make_receipt(
    tool_name="hold_seat",
    tool_args={"flight": "QF11", "seat": "14A", "hold_minutes": 30},
    tool_result={"hold_id": "H8472", "expires_at": "2026-06-15T15:00:00Z"},
    sequence=1,
    previous_receipt_hash=receipt_hash(r0),
    signing_key=signing_key,
    verify_key=verify_key,
)

r2 = make_receipt(
    tool_name="confirm_booking",
    tool_args={"hold_id": "H8472", "payment_token": "tok_redacted"},
    tool_result={"booking_ref": "CT-09182", "status": "confirmed"},
    sequence=2,
    previous_receipt_hash=receipt_hash(r1),
    signing_key=signing_key,
    verify_key=verify_key,
)

chain = [r0, r1, r2]
for i, r in enumerate(chain):
    print(f"Receipt {i}: tool={r['tool_name']}, prev={r['previous_receipt_hash']}")

Receipt 0: tool=lookup_flights, prev=None
Receipt 1: tool=hold_seat, prev=sha256:af8b9159b25728881ce6a7166429f4451b9fa869a337453d52228f12131192e7
Receipt 2: tool=confirm_booking, prev=sha256:b1634969232be232f53283251cc9307b443a296909d9e74c2b0101b7f4dc679e


In [12]:
def verify_chain(chain: list, trusted_public_key: str) -> list[dict]:
    """
    Verify a sequence of receipts:
      1. Each receipt's signature must verify.
      2. Each receipt must carry previous_receipt_hash: null for the genesis, and
         the previous receipt's hash for every later receipt.
      3. Each sequence number must be an integer equal to the receipt's zero-based
         position in the chain.
    Returns a list of per-receipt result dicts. A malformed entry fails its own
    checks; it does not stop the rest of the chain from being checked.
    """
    results = []
    for i, receipt in enumerate(chain):
        is_receipt = isinstance(receipt, dict)
        sig_ok = verify_receipt(receipt, trusted_public_key)

        # The link field must be present: .get() alone would read a missing field as None.
        has_link = is_receipt and "previous_receipt_hash" in receipt
        link = receipt.get("previous_receipt_hash") if is_receipt else None
        if i == 0:
            chain_ok = has_link and link is None
        else:
            try:
                chain_ok = has_link and link == receipt_hash(chain[i - 1])
            except Exception:
                chain_ok = False  # the previous entry could not be hashed

        # type() rather than isinstance(): bool is a subclass of int, and True == 1.
        sequence = receipt.get("sequence") if is_receipt else None
        seq_ok = type(sequence) is int and sequence == i
        shape_ok = is_receipt and isinstance(receipt.get("tool_name"), str)

        results.append({
            "index": i,
            "tool": str(receipt.get("tool_name", "?")) if is_receipt else "(not a receipt)",
            "signature_valid": sig_ok,
            "chain_link_valid": chain_ok,
            "sequence_valid": seq_ok,
            "overall_valid": sig_ok and chain_ok and seq_ok and shape_ok,
        })
    return results

for r in verify_chain(chain, public_key_b64):
    status = "VALID" if r["overall_valid"] else "INVALID"
    print(f"Receipt {r['index']} ({r['tool']:>18}): {status}")

Receipt 0 (    lookup_flights): VALID
Receipt 1 (         hold_seat): VALID
Receipt 2 (   confirm_booking): VALID


ယခုစားပွဲအလယ်ရှိ လက်ခံလွှဲစာရင်းကိုပြောင်းလဲ၍ ချိတ်ဆက်မှုကို ကွဲချ ပြီး ပြန်စစ် ဆောင်ရွက်ပါ။ ပြောင်းလဲထားသော လက်ခံလွှဲစာရင်းသည် ဆံ့ကြိုး စစ်ဆေးမှု၌ မအောင်မြင်ပါ၊ ထို့အပြင် နောက်တစ်ခုသော လက်ခံလွှဲစာရင်းသည် ချိတ်ဆက်မှု စစ်ဆေးမှု၌ မအောင်မြင်ပါ (အကြောင်းမှာ ၎င်း၏ `previous_receipt_hash` သည် ပြင်ဆင်ထားသော အလယ် လက်ခံလွှဲစာရင်း၏ hash နှင့် မကိုက်ညီတော့ပါ)။


In [13]:
# Tamper with the middle receipt: change the hold duration to something
# more permissive than was actually authorized.
tampered_chain = [copy.deepcopy(r) for r in chain]
tampered_chain[1]["tool_args_hash"] = sha256_canonical(
    {"flight": "QF11", "seat": "14A", "hold_minutes": 9999}
)

for r in verify_chain(tampered_chain, public_key_b64):
    status = "VALID" if r["overall_valid"] else "INVALID"
    why = ""
    if not r["overall_valid"]:
        reasons = []
        if not r["signature_valid"]:
            reasons.append("signature")
        if not r["chain_link_valid"]:
            reasons.append("chain link")
        if not r["sequence_valid"]:
            reasons.append("sequence")
        why = " (failed: " + ", ".join(reasons) + ")"
    print(f"Receipt {r['index']} ({r['tool']:>18}): {status}{why}")

Receipt 0 (    lookup_flights): VALID
Receipt 1 (         hold_seat): INVALID (failed: signature)
Receipt 2 (   confirm_booking): INVALID (failed: chain link)


ချည်းနှီးချက် 0 သည် သွယ်တန်းအား စစ်ဆေးမှုကို ဆက်လက်ပြုလုပ်နိုင်သည် (၎င်းအား မပြုပြင်ထားပြီး မည်သည့်ကြိုတင်ထားသည့် အရာမရှိပါ။) ချည်းနှီးချက် 1 သည် အမှတ်အသားစစ်ဆေးမှုတွင် မအောင်မြင်ပါ၊ ဤသည်မှာ ကျွန်ုပ်တို့ `tool_args_hash` ကိုပြောင်းလဲထားသည်။ ချည်းနှီးချက် 2 သည် သတ်မှတ်ထားသည့် ချည်းနှီးမှုလွှာစစ်ဆေးမှုတွင် မအောင်မြင်ပါ၊ ဤသည်တွင် ၎င်း၏ `previous_receipt_hash` ကို မူလ (ယခုပြောင်းလဲထားသော) ချည်းနှီးချက် 1 အပေါ် အခြေခံပြီး တွက်ချက်ခဲ့သည်။

အပစ်မှတ်တစ်ဦးမှ ပြင်ဆင်ပြီးသော ချည်းနှီးချက် 1 ကို ထပ်မံ အမှတ်အသားရေးထိုးလျှင်လည်း၊ ထိုမှတ်တမ်းသစ်သည် ထုတ်ပေးသူ၏သော့မှာ မရှိစွာ ရရှိထားခြင်းမရှိသည့်အထိ ဤသို့ လုပ်ဆောင်လာပါက မအောင်မြင်ပါ (၎င်းတို့၏ကိုယ်ပိုင်သော့ဖြင့် ထပ်မံ အမှတ်အသားရေးထိုးခြင်းသည် အပိုင်း 2 တွင် ပြသခဲ့သည့်အတိုင်း မအောင်မြင်ပါ)၊ ချည်းနှီးချက် 2 တွင် ချည်းနှီးမှုလွှာ မကိုက်ညီမှုသည် လိမ်လည်မှုကို ဖော်ထုတ်ပါလိမ့်မည်။ ပြောင်းလဲမှုကို ဖုံးကွယ်လိုလျှင် အပစ်မှတ်တစ်ဦးသည် ပြောင်းလဲခဲ့သောအမှတ်အသားမှ စတင်၍ ချည်းနှီးချက်အားလုံးကို ထပ်မံ အမှတ်အသားရေးထိုးရမည်ဖြစ်ပြီး၊ ၎င်းသည် ထုတ်ပေးသူ၏ကိုယ်ပိုင်သော့ကို စွဲထိန်းထားရမည်ဖြစ်သည်။


## အပိုင်း ၄: ကိရိယာခေါ်ဆိုမှုကို ရရှိမှတ်ချက်ဖြင့် အုပ်ပုံထုပ်ခြင်း  

လက်တွေ့ တပ်ဆင်မှုမှာ သက်ဆိုင်ရာ agent ဖန်တီးသူတိုင်းက `make_receipt` ကို ခေါ်ရန် တိုင်းတာရမည် မလိုပါ။ ကိရိယာ ခေါ်ဆိုမှု အတွက် ရရှိမှတ်ချက် လက်မှတ်ရေးထိုးခြင်းကို အလိုအလျောက် ဖြစ်စေချင်ပါတယ်။  

ဒီမှာ အလွန်ရိုးရှင်းတဲ့ ပုံစံတစ်ခုရှိသည်။ ခေါ်နိုင်သည့် ကိရိယာလုပ်ဆောင်ချက်ကို ယူပြီး ရရှိမှတ်ချက် ထုတ်ပေးသော ဗားရှင်းကို ပြန်ပေးပို့သော ဝါရပ်ယာ ကလပ်စ်တစ်ခုဖြစ်သည်။ ၎င်းသည် Microsoft Agent Framework (`agent_framework.foundry`) တို့ အပါအဝင် agent အဆောက်အအုံ ဘာ Framework ဖြစ်သည့် နေရာမဆို လိုက်ဖက်သည်။  

သင်မှာ Microsoft Foundry ပရောဂျက် မတည်ဆောက်ထားလျှင် ဖောက်ပြန်နေသော ရှုပ်ထွေးမှုကို အောက်ပါ ဒေသတွင်း နမူနာ က ကိုင်တွယ်ပြသထားသည်။  


In [14]:
class ReceiptedTool:
    """
    Wraps a tool function so every invocation produces a signed receipt.
    Receipts are appended to a chain held by this object.

    Accepts both positional and keyword arguments. The receipt's
    tool_args field records args (as a list) and kwargs (as a dict)
    so the canonical hash binds to whichever the caller supplied.
    """

    def __init__(self, name: str, fn, signing_key, verify_key, agent_id: str, policy_id: str):
        self.name = name
        self.fn = fn
        self.signing_key = signing_key
        self.verify_key = verify_key
        self.agent_id = agent_id
        self.policy_id = policy_id
        self.receipts: list = []

    def __call__(self, *args, **kwargs):
        result = self.fn(*args, **kwargs)
        previous_hash = receipt_hash(self.receipts[-1]) if self.receipts else None
        receipt = make_receipt(
            tool_name=self.name,
            tool_args={"args": list(args), "kwargs": kwargs},
            tool_result=result,
            sequence=len(self.receipts),
            previous_receipt_hash=previous_hash,
            signing_key=self.signing_key,
            verify_key=self.verify_key,
            agent_id=self.agent_id,
            policy_id=self.policy_id,
        )
        self.receipts.append(receipt)
        return result

In [15]:
# Example tool: a mock flight lookup. In a real Microsoft Agent Framework deployment,
# this would be a function passed to FoundryChatClient as a tool.
def mock_lookup_flights(origin: str, destination: str, departure_date: str) -> list:
    return [
        {"flight": "QF11", "price": 1850, "stops": 0},
        {"flight": "UA864", "price": 1620, "stops": 1},
    ]

# Wrap it with receipt signing.
receipted_lookup = ReceiptedTool(
    name="lookup_flights",
    fn=mock_lookup_flights,
    signing_key=signing_key,
    verify_key=verify_key,
    agent_id="contoso-travel-bot",
    policy_id="contoso-travel-policy-v3",
)

# Use the wrapped tool exactly like the original.
results_a = receipted_lookup(origin="SYD", destination="LAX", departure_date="2026-06-15")
results_b = receipted_lookup(origin="SYD", destination="NRT", departure_date="2026-07-02")
results_c = receipted_lookup(origin="MEL", destination="SIN", departure_date="2026-08-10")

print(f"Tool was called {len(receipted_lookup.receipts)} times.")
print(f"Each call produced a signed receipt linked to the previous one.")
print()

for r in verify_chain(receipted_lookup.receipts, public_key_b64):
    status = "VALID" if r["overall_valid"] else "INVALID"
    print(f"Receipt {r['index']} ({r['tool']}): {status}")


Tool was called 3 times.
Each call produced a signed receipt linked to the previous one.

Receipt 0 (lookup_flights): VALID
Receipt 1 (lookup_flights): VALID
Receipt 2 (lookup_flights): VALID


### Microsoft Agent Framework နှင့် ပေါင်းစည်းခြင်း

အထက်ပါ `ReceiptedTool` wrapper သည် framework-agnostic ဖြစ်သည်။ Microsoft Agent Framework ဖြင့် တည်ဆောက်ထားသော agent အတွင်း၌ အသုံးပြုရန်အတွက်၊ ထုပ်လွှင့်ထားသော function ကို tool အဖြစ် မှတ်ပုံတင်ပါ။ အညွှန်း (သင်သည် mock ကို အမှန်တကယ် Microsoft Foundry tool မှတ်ပုံတင်မှုဖြင့် အစားထိုးပါမည်)။

```python
# Pseudocode showing the integration shape.
# import os
# from agent_framework.foundry import FoundryChatClient
# from azure.identity import AzureCliCredential
#
# provider = FoundryChatClient(
#     project_endpoint=os.environ["AZURE_AI_PROJECT_ENDPOINT"],
#     model=os.environ["AZURE_AI_MODEL_DEPLOYMENT_NAME"],
#     credential=AzureCliCredential(),
# )
# agent = provider.as_agent(
#     instructions="You are a Contoso Travel agent ...",
#     tools=[receipted_lookup],   # the wrapped tool, not the raw function
# )
# response = agent.run("Find flights from Sydney to Los Angeles in June.")
#
# # After the run, every tool call the agent made has a signed receipt:
# audit_chain = receipted_lookup.receipts
```

Agent framework သည် လက်ခံခြင်းစာရင်းများအကြောင်း မည်သည့်အချက်အလက်မျှ မလိုအပ်ပါ။ လက်ခံခြင်းစာရင်း လက်မှတ်ရေးထိုးခြင်းသည် tool အပေါ်တွင် ထုပ်လွှင့်ထားပြီး framework ထဲသို့ မပေါင်းစည်းထားပါ။ ၎င်းသည် agent ကို ပြန်ရေးသားစရာမလိုဘဲ ရှိပြီးသား agent ကုဒ်များတွင် provenance ကို ထည့်သွင်းပေးနိုင်သည့်နည်းလမ်းဖြစ်သည်။


## ပြန်လည်သုံးသပ်ခြင်းနှင့် စိန်ခေါ်မှုတိုက်

သင်သည်:

- Ed25519 key အဖုံကို ထုတ်လုပ်ခြင်း။
- ကိုယ်စားလှယ်ကိရိယာခေါ်ဆိုမှုအတွက် လက်မှတ်ထိုးပြီး ရရှိသော receipt ကို တည်ဆောက်ခြင်း။
- ထုတ်ပြန်သူ၏ public key တစ်ခုသာ အသုံးပြု၍ receipt ကို အွန်လိုင်းမရှိဘဲစစ်ဆေးခြင်း။
- Receipt ကို ပြုပြင်လုပ်ဆောင်ပြီး စစ်ဆေးမှု မအောင်မြင်သည်ကို လေ့လာတွေ့ရှိခြင်း။
- ဓာတ်ပုံသဏ္ဍာန် သုံးခုပါတဲ့ receipt အစဉ်လိုက် ဆက်စပ်မှုကို တည်ဆောက်ခြင်း။
- အစဉ်မှာ အလယ်ပိုင်းမှာ ပြုပြင်ဆင်ခြင်မှုဖြစ်ပြီး လက်မှတ် မအောင်မြင်မှု နှင့် အစဉ်လိုက် ဆက်စပ်မှု မအောင်မြင်မှုကို တွေ့ရှိခြင်း။
- ကိရိယာလုပ်ဆောင်မှုကို အလိုအလျောက် လက်မှတ်ထိုးမှုဖြင့် ထုပ်ပိုးခြင်း။

**စိန်ခေါ်မှုပိုင်း.** Receipt schema ကို `request_id` ကွက်လပ် (လွှဲပြောင်းမည့် UUID) ဖြင့် တိုးချဲ့ပါ။ `make_receipt` မှာပါဝင်စေပြီး receipt များကို အဆုံးအထိတွက်ချက်မှု အတည်ပြုပါ။ ထို့နောက် လက်မှတ်ထိုးပြီးနောက် ဖீလ်ကို ပြင်ဆင်ပြီး အတည်ပြုမှုမအောင်မြင်ဘဲ ပိုမိုတည်ရှိမှု စစ်ဆေးပါ။ ဒါက canonical encoding ၏ တစ်လုံးချင်း bytes အားလုံးကြောင့် လက်မှတ်ထိုးမှုမှာ ဘယ်လိုပိုင်းဆိုင်ရာ သက်ရောက်မှုရှိသည်ကို နက်ရှိုင်းစွာနားလည်ရန် လိုအပ်စေသည်။

**အရေးကြီးသော နယ်နိမိတ်။** သေချာထားသော key နှင့် ဓာတ်ပုံခေါင်းချဲ့ခြင်း အကြောင်းအရာအောက်တွင် receipt များသည် လက်မှတ်ပိုင်ခွင့်၊ အကြောင်းအရာတည်ငြိမ်မှု၊ မတိုင်မီရဲ႕ ဆက်စပ်မှုကို အတည်ပြုပေးသည်။ အကောင်အထည်မပေါ်ခြင်း၊ မူရင်း အမှန်တရား၊ ယုံကြည်စိတ်ချရသောအချိန်၊ ပြည့်စုံမှု၊ မူဝါဒ အကဲဖြတ်ခြင်း သို့မဟုတ် တိကျမှုကို မအတည်ပြုပါ။ Receipt များသည် အခြေခံကျသောအရာဖြစ်သည်။ အုပ်ချုပ်မှုဆိုတာ သင်တည်ဆောက်လိုက်သော စနစ်ဖြစ်သည်။

အဆိုပါ နယ်နိမိတ်နှင့်အတူ သင်ခန်းစာ README ကို ထပ်မံဖတ်ရှုပါ။ Receipt များနှင့်အတူ အဖွဲ့များ မကြာခဏ ဖြစ်ပေါ်သော အမှားမှာ "ကျွန်တော်တို့ Receipt ရှိတယ်" ဆိုသည်မှာ "ကျွန်တော်တို့ အုပ်ချုပ်ခံထားရတယ်" ဖြစ်သည်ဟု မှတ်ယူခြင်းဖြစ်သည်။ မဟုတ်ပါ။ Receipt များသည် ကိုယ်စားလှယ်၏ အပြုအမူကို စိစစ်နိုင်စေသည်။ ဒါဟာ တိကျမှုကို မပေးစွမ်းပါ။ 


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ပြောကြားချက်**
ဤစာတမ်းကို AI ဘာသာပြန်ဝန်ဆောင်မှု [Co-op Translator](https://github.com/Azure/co-op-translator) အသုံးပြု၍ ဘာသာပြန်ထားပါသည်။ ကျွန်ုပ်တို့သည် တိကျမှန်ကန်မှုအတွက် ကြိုးပမ်းနေသော်လည်း၊ စက်ကိရိယာဘာသာပြန်ခြင်းများတွင် အမှားများ သို့မဟုတ် မှားယွင်းချက်များ ပါဝင်နိုင်ကြောင်း သတိပြုပါရန် လိုအပ်ပါသည်။ မူလစာတမ်းကို မူရင်းဘာသာဖြင့်သာ ယုံကြည်စိတ်ချရသော အချက်အလက်အဖြစ် သတ်မှတ်သင့်သည်။ အရေးကြီးသည့် သတင်းအချက်အလက်များအတွက် ပရော်ဖက်ရှင်နယ် လူသားဘာသာပြန်သူဝန်ဆောင်မှုကို အကြံပြုပါသည်။ ဤဘာသာပြန်ချက်ကို အသုံးပြုခြင်းမှ ဖြစ်ပေါ်လာသော နားလည်မှုကွာခြားမှုများ သို့မဟုတ် မမှန်ကန်သော အသုံးပြုမှုများအတွက် ကျွန်ုပ်တို့ တာဝန်မခံပါ။
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
